# Build dataset

In [ ]:
# Library Imports
import os
import re
import numpy as np
import pandas as pd
from tabulate import tabulate

## Data loader from Excel file

In [ ]:
# ============================================================
# FILE PATHS
# ============================================================
data_dir = os.path.join(
    os.path.dirname(os.path.abspath(__file__)) 
    if "__file__" in globals() else os.getcwd(), 
    "data"
)
files = {
    "Active":   os.path.join(data_dir, "Active_companies.xlsx"),
    "Bankrupt": os.path.join(data_dir, "Bankrupt_companies.xlsx"),
}

# ============================================================
# HELPER — CLEAN RAW DATAFRAME ON LOAD
# ============================================================
# Applied once per file immediately after reading from disk.
# Handles: unnamed index columns, incorporation year parsing,
# NACE code zero-padding, financial column type conversion.

def clean_df(df):
    df = df.copy()

    # Drop unnamed index columns Orbis sometimes exports
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])

    # Incorporation year — handle both Orbis export formats:
    #   "Date of incorporation" → datetime string → extract year
    #   "Year of incorporation" → integer directly
    if "Date of incorporation" in df.columns:
        df["Year of incorporation"] = (
            pd.to_datetime(df["Date of incorporation"], errors="coerce")
            .dt.year.astype("Int16")
        )
        df = df.drop(columns=["Date of incorporation"])
    elif "Year of incorporation" in df.columns:
        df["Year of incorporation"] = (
            pd.to_numeric(df["Year of incorporation"], errors="coerce")
            .astype("Int16")
        )

    # NACE code — preserve as zero-padded 4-character string
    # e.g. integer 711 → "0711" to avoid losing leading zeros
    nace_col = "NACE Rev. 2, core code (4 digits)"
    if nace_col in df.columns:
        df[nace_col] = df[nace_col].astype(str).str.zfill(4)

    # Financial columns — any column ending with a 4-digit year
    # converted to Float64 (nullable float, supports NaN)
    fin_cols = [c for c in df.columns if re.search(r"\d{4}$", c)]
    df[fin_cols] = df[fin_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")

    return df.copy()

# ============================================================
# HELPER — PRINT DATAFRAME STRUCTURE
# ============================================================
def print_df_structure(name, df):
    rows = []
    for i, col in enumerate(df.columns):
        non_null = df[col].notna().sum()
        null     = df[col].isna().sum()
        rows.append([i, col, str(df[col].dtype), f"{non_null:,}", f"{null:,}"])

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Rows: {len(df):,}    Columns: {len(df.columns)}")
    print()
    print(tabulate(
        rows,
        headers=["#", "Column", "Dtype", "Non-Null", "Null"],
        tablefmt="github",
        numalign="right",
        stralign="left",
    ))
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\n  Memory usage: {mem_mb:.2f} MB\n")

# ============================================================
# LOAD FILES
# ============================================================
df_active   = None
df_bankrupt = None

for label, path in files.items():
    if not os.path.exists(path):
        print(f"File not found: {path}")
        continue
    try:
        df = pd.read_excel(path)
        df = clean_df(df)
        if label == "Active":
            df_active = df
            df_name   = "df_active"
        else:
            df_bankrupt = df
            df_name     = "df_bankrupt"
        print(f"Loaded: {df_name} ({os.path.basename(path)})")
        print_df_structure(df_name, df)
    except Exception as e:
        print(f"Error loading {path}: {e}")

# ============================================================
# PRESERVE RAW COPIES FOR IDEMPOTENT RE-RUNS
# ============================================================
# Processing blocks always reset from these _raw copies,
# meaning they can be re-run without reloading from disk
# and without accumulating side effects from previous runs.

if df_active is not None:
    df_active_raw = df_active.copy()

if df_bankrupt is not None:
    df_bankrupt_raw = df_bankrupt.copy()

print("✓ Raw copies saved: df_active_raw, df_bankrupt_raw")

## Data Preprocessing & panel data structure 

In [ ]:
# ============================================================
# 0. RESET — ensure idempotent execution on re-runs
# ============================================================

df_active   = df_active_raw.copy()
df_bankrupt = df_bankrupt_raw.copy()


# ============================================================
# 1. MERGE ACTIVE AND BANKRUPT COMPANIES
# ============================================================
# Assign binary bankruptcy label:
#   0 = active / solvent company
#   1 = bankrupt / dissolved (bankruptcy) company
# Then concatenate into one unified DataFrame.
# ignore_index=True resets row index to 0..N continuously.

df_active["Bankrupt"]   = 0
df_bankrupt["Bankrupt"] = 1

df_analysis = pd.concat([df_active, df_bankrupt], ignore_index=True)

print(f"After concat: {len(df_analysis):,} rows x {len(df_analysis.columns)} columns")
print(f"  Active:   {(df_analysis['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt: {(df_analysis['Bankrupt'] == 1).sum():,}")


# ============================================================
# 1b. REMOVE EXACT DUPLICATES AND ASSIGN A UNIQUE COMPANY ID
# ============================================================
# Company names are not unique: different companies can share a
# name, and grouping by name would merge their rows (mixing their
# financials and labels). Every Orbis row is therefore given its
# own company_id, which is used as the company key from here on.
# The name is kept for display only.
#
# Rows that are identical in every column are the same company
# exported twice — those are dropped first, so they do not become
# two "companies".

name_col = "Company name Latin alphabet"

n_before = len(df_analysis)
df_analysis = df_analysis.drop_duplicates().reset_index(drop=True)
n_exact_dups = n_before - len(df_analysis)

df_analysis.insert(0, "company_id", df_analysis.index)

shared = df_analysis[df_analysis[name_col].duplicated(keep=False)]
n_shared_names = shared[name_col].nunique()
n_cross_label  = int((shared.groupby(name_col)["Bankrupt"].nunique() > 1).sum())

print(f"\nDuplicate check:")
print(f"  Exact duplicate rows removed:         {n_exact_dups:,}")
print(f"  Names shared by 2+ companies:         {n_shared_names:,} "
      f"({len(shared):,} companies)")
print(f"  ...of which in both active & bankrupt: {n_cross_label:,}")
print(f"  Companies after de-duplication:       {len(df_analysis):,} "
      f"(active {(df_analysis['Bankrupt'] == 0).sum():,}, "
      f"bankrupt {(df_analysis['Bankrupt'] == 1).sum():,})")


# ============================================================
# 2. MAP NACE CODES TO INDUSTRY SECTIONS
# ============================================================
# Orbis exports 4-digit NACE Rev. 2 codes (e.g. 4711 = supermarkets).
# We group these into broader section-level categories using the
# first 2 digits (division level) per the official Eurostat
# NACE Rev. 2 classification structure.
#
# Sections K, O, P, Q, D, T, U were excluded upstream in Orbis
# via section letter filters and should not appear here.
# Returns "Unknown" for any unmapped code for data quality detection.

nace_section_map = {
    range(1,  4):  "A - Agriculture, forestry, fishing",
    range(5,  10): "B - Mining and quarrying",
    range(10, 34): "C - Manufacturing",
    range(36, 40): "E - Water supply, waste management",
    range(41, 44): "F - Construction",
    range(45, 48): "G - Wholesale and retail trade",
    range(49, 54): "H - Transport and storage",
    range(55, 57): "I - Accommodation and food service",
    range(58, 64): "J - Information and communication",
    range(68, 69): "L - Real estate",
    range(69, 76): "M - Professional, scientific, technical",
    range(77, 83): "N - Administrative and support services",
    range(90, 94): "R - Arts, entertainment, recreation",
    range(94, 97): "S - Other service activities",
}

def nace_to_section(code):
    """
    Maps a 4-digit NACE Rev. 2 code to its section label.
    Uses first 2 digits (division) for lookup.
    Returns 'Unknown' for unmatched codes.
    """
    try:
        division = int(str(code)[:2])
        for r, section in nace_section_map.items():
            if division in r:
                return section
        return "Unknown"
    except:
        return "Unknown"

df_analysis["NACE Section"] = (
    df_analysis["NACE Rev. 2, core code (4 digits)"]
    .apply(nace_to_section)
)

unknown_nace = df_analysis[df_analysis["NACE Section"] == "Unknown"]
if len(unknown_nace) > 0:
    print(f"\nUnknown NACE codes ({len(unknown_nace)} companies):")
    print(unknown_nace["NACE Rev. 2, core code (4 digits)"].value_counts().head(10))
else:
    print("✓ All NACE codes mapped successfully")


# ============================================================
# 3. CLEAN YEAR OF INCORPORATION
# ============================================================
# Force to numeric to handle any non-numeric Orbis export values
# (empty strings, "n.a." etc.) → converted to NaN safely.
# Int64 (nullable integer) used instead of int64 to support NaN.
# Used later to compute company age at each observation year.

df_analysis["Year of incorporation"] = pd.to_numeric(
    df_analysis["Year of incorporation"], errors="coerce"
).astype("Int64")


# ============================================================
# 4. REORDER COLUMNS — IDENTIFIERS FIRST
# ============================================================
# Place identifier and label columns at the front for readability.
# Financial time-series columns follow after.
# Cosmetic only — does not affect downstream modeling.

front_cols = [
    "company_id",
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

remaining_cols = [c for c in df_analysis.columns if c not in front_cols]
df_analysis = df_analysis[front_cols + remaining_cols]


# ============================================================
# 5. RESHAPE WIDE → PANEL (LONG) FORMAT
# ============================================================
# Orbis exports in wide format: one row per company, with year-
# suffixed columns for each financial variable:
#   "Total assets EUR 2022", "Total assets EUR 2023", ...
#
# Panel (long) format is required for time-series feature
# engineering: one row per company-year combination, one column
# per financial variable.
#
# Reshape approach:
#   Step A — melt:   wide → long (one row per company × column)
#   Step B — parse:  extract year and metric name from column name
#   Step C — pivot:  long → panel (one row per company × year)

id_cols = [
    "company_id",
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

fin_cols = [c for c in df_analysis.columns if c not in id_cols]

# ── Step A: Melt wide → long ─────────────────────────────────
# Result: one row per (company × financial column)
# col_raw = original column name e.g. "Total assets EUR 2022"
# value   = financial figure in EUR
df_long = df_analysis.melt(
    id_vars=id_cols,
    value_vars=fin_cols,
    var_name="col_raw",
    value_name="value",
)

# ── Step B: Parse year and metric name ───────────────────────
# Year:   extract trailing 4-digit year from column name
# Metric: strip "EUR YYYY" suffix to get clean variable name
df_long["Year"] = (
    df_long["col_raw"]
    .str.extract(r"(\d{4})$")
    .astype("Int64")
)

df_long["Metric"] = (
    df_long["col_raw"]
    .str.replace(r"\s*EUR\s*\d{4}$", "", regex=True)
    .str.strip()
)

df_long = df_long.drop(columns="col_raw")

# ── Step C: Pivot long → panel ───────────────────────────────
# Result: one row per (company, year)
# Each metric becomes its own column
# aggfunc="first" handles any accidental duplicates
df_panel = (
    df_long
    .pivot_table(
        index=id_cols + ["Year"],
        columns="Metric",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)
df_panel.columns.name = None


# ============================================================
# 6. DROP FULLY EMPTY ROWS
# ============================================================
# After reshaping, rows exist for every company × year pair even
# when Orbis had no financial data for that year (all NaN values)
# — e.g. a bankrupt company's years after it stopped filing.
# These empty rows add no information.
#
# Strategy: drop rows where ALL financial metric columns are NaN.
# Rows with partial data are retained — XGBoost handles missing
# values natively so partial observations are still useful.

metric_cols = [c for c in df_panel.columns if c not in id_cols + ["Year"]]

before = len(df_panel)
df_panel = df_panel.dropna(subset=metric_cols, how="all").reset_index(drop=True)
after  = len(df_panel)

print(f"\nDropped {before - after:,} fully empty rows")
print(f"df_panel: {len(df_panel):,} rows x {len(df_panel.columns)} columns")
print(f"  Years covered: {df_panel['Year'].min()} – {df_panel['Year'].max()}")
print(f"  Active rows:   {(df_panel['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt rows: {(df_panel['Bankrupt'] == 1).sum():,}")
print(f"  Companies:     {df_panel['company_id'].nunique():,} "
      f"(active {df_panel.loc[df_panel['Bankrupt'] == 0, 'company_id'].nunique():,}, "
      f"bankrupt {df_panel.loc[df_panel['Bankrupt'] == 1, 'company_id'].nunique():,})")

# pivot_table drops rows whose identifier columns contain NaN
# (e.g. a missing year of incorporation) — report any such loss
lost = set(df_analysis["company_id"]) - set(df_panel["company_id"])
if lost:
    print(f"  Note: {len(lost)} company(ies) dropped in reshape "
          f"(missing identifier value or no financial data)")


# ============================================================
# 7. COMPUTE COMPANY AGE AT OBSERVATION YEAR
# ============================================================
# Company age = observation year − year of incorporation.
# This gives the company's age in years at the time each
# financial statement was filed — not today's age.
#
# Example: company incorporated 1997, observation year 2022
#          → company age = 25 years at time of filing
#
# Using the observation Year column (panel year) rather than
# current year ensures age is correct for each row in the panel.
# A company with data in 2022, 2023, and 2024 will show ages
# 25, 26, and 27 respectively — reflecting its actual age when
# each set of financials was reported.
#
# Int64 used to support NaN for the one company missing
# Year of incorporation in the raw Orbis export.

df_panel["Company Age"] = (
    df_panel["Year"] - df_panel["Year of incorporation"]
).astype("Int64")

# Sanity checks
age_min = df_panel["Company Age"].min()
age_max = df_panel["Company Age"].max()
age_neg = (df_panel["Company Age"] < 0).sum()
age_null = df_panel["Company Age"].isna().sum()

print(f"Company Age range:     {age_min} – {age_max} years")
print(f"Negative ages (error): {age_neg}")   # should be 0
print(f"Missing ages:          {age_null}")  # should be ≤ 1

# Move Company Age next to Year of incorporation for readability
cols = df_panel.columns.tolist()
year_inc_idx = cols.index("Year of incorporation")
cols.remove("Company Age")
cols.insert(year_inc_idx + 1, "Company Age")
df_panel = df_panel[cols]

print_df_structure("df_panel", df_panel)

# Feature Engineering

In [ ]:
# ============================================================
# 8. FEATURE ENGINEERING
# ============================================================
# Derives financial ratios, distress indicators, and trend
# features from the 5 raw Orbis variables. All engineered
# features are prefixed with "FE_" to clearly distinguish them
# from raw Orbis variables throughout the pipeline.
#
# Features computed:
#   FE_total_liabilities  — derived balance sheet item
#   FE_non_current_assets — derived balance sheet item
#   FE_debt_ratio         — solvency / leverage
#   FE_current_ratio      — short-term liquidity
#   FE_roa                — profitability relative to assets
#   FE_negative_equity    — binary distress flag
#   FE_negative_pl        — binary distress flag
#   FE_assets_yoy_growth  — year-over-year asset trajectory
#   FE_equity_yoy_change  — year-over-year equity trajectory
#
# Division safety:
#   Values are in EUR. Where a denominator is effectively zero
#   (abs < €1,000), the ratio is set to NaN rather than using
#   epsilon inflation. This is more honest — a near-zero
#   denominator means the ratio is undefined, not astronomically
#   large. XGBoost handles NaN natively so no information is lost.
#
# Trend features:
#   YoY features are NaN for a company's earliest observed
#   year — no prior year available to diff against. Expected
#   and handled natively by XGBoost.

# Work on a copy — df_panel remains unchanged
df_features = df_panel.copy()
df_features = df_features.sort_values(
    ["company_id", "Year"]
).reset_index(drop=True)

# Safe division helper — returns NaN when denominator is
# effectively zero rather than inflating with epsilon
def safe_divide(numerator, denominator, threshold=1_000):
    """
    Divides numerator by denominator element-wise.
    Returns NaN where abs(denominator) < threshold.
    Values are in EUR: a denominator below €1,000 (e.g. near-zero
    current liabilities) is treated as undefined rather than
    producing extreme ratios.
    Converts Float64 (pandas nullable) to float64 (numpy)
    before comparison to avoid pd.NA boolean ambiguity.
    Returns pandas Series to preserve index alignment.
    """
    # Convert to numpy float64 — resolves pd.NA boolean ambiguity
    # that occurs with pandas Float64 nullable dtype
    num = numerator.astype("float64")
    den = denominator.astype("float64")

    result = np.where(
        den.abs() < threshold,
        np.nan,
        num / den
    ).astype(float)

    return pd.Series(result, index=numerator.index)
# ── A: Derived balance sheet items ───────────────────────────
# Accounting identity: Assets = Equity + Liabilities
# Total liabilities was not directly exported from Orbis —
# derived here from the two available balance sheet components.

df_features["FE_total_liabilities"] = (
    df_features["Total assets"]
    - df_features["Shareholders funds"]
)

df_features["FE_non_current_assets"] = (
    df_features["Total assets"]
    - df_features["Current assets"]
)

# ── B: Solvency / leverage ────────────────────────────────────
# Proportion of total assets financed by debt.
# Values > 1 occur when equity is negative — itself a distress
# signal captured separately by FE_negative_equity flag.
# NaN when total assets is effectively zero.

df_features["FE_debt_ratio"] = safe_divide(
    df_features["FE_total_liabilities"],
    df_features["Total assets"]
)

# ── C: Liquidity ──────────────────────────────────────────────
# Current ratio < 1: current liabilities exceed current assets
# meaning the company cannot cover short-term obligations from
# liquid assets alone. Classic early warning indicator.
# NaN when current liabilities is effectively zero — a company
# with no current liabilities is not distressed, but the ratio
# is undefined rather than infinite.

df_features["FE_current_ratio"] = safe_divide(
    df_features["Current assets"],
    df_features["Current liabilities"]
)

# ── D: Profitability ──────────────────────────────────────────
# Return on assets: profit generated per unit of total assets.
# Equivalent to Altman Z-score variable X3.
# Strongest single predictor in most bankruptcy studies.
# Negative ROA = loss-making relative to asset base.
# NaN when total assets is effectively zero.

df_features["FE_roa"] = safe_divide(
    df_features["P/L before tax"],
    df_features["Total assets"]
)

# ── E: Binary distress flags ──────────────────────────────────
# Hard threshold indicators of specific danger zones.
# Int64 (nullable integer) used — Float64 columns contain
# pd.NA which cannot be cast to plain int.
#
# FE_negative_equity: shareholders funds < 0 means accumulated
#   losses have exceeded paid-in capital — the company is
#   technically insolvent under Finnish Companies Act (OYL
#   chapter 20 section 1). A board is legally required to
#   take action when equity falls below half of share capital.
#
# FE_negative_pl: company recorded a pre-tax loss this year.
#   One loss year may be recoverable — in combination with
#   negative equity and high debt ratio it signals severe
#   multi-dimensional distress.

df_features["FE_negative_equity"] = (
    (df_features["Shareholders funds"] < 0).astype("Int64")
)

df_features["FE_negative_pl"] = (
    (df_features["P/L before tax"] < 0).astype("Int64")
)

# ── F: Year-over-year trend features ─────────────────────────
# Capture trajectory of financial deterioration across the
# 2022–2024 panel window.
#
# Grouping by a unique company key ensures changes are computed
# only within the same company — never across companies. The key
# is company_id (unique per Orbis row); company name is not used
# because different companies can share a name.
#
# "Prior year" means the previous calendar year (Year − 1), not
# simply the previous row: if a company has a gap (e.g. 2022 and
# 2024 but no 2023), the change is left as NaN rather than
# labelling a two-year change as a yearly one.
#
# FE_assets_yoy_growth:
#   Proportional asset change relative to prior-year total assets.
#   Negative growth = shrinking asset base, distress signal.
#
# FE_equity_yoy_change:
#   Change in shareholders funds, scaled by prior-year total assets.
#   Not a % change in equity — that is unstable when equity is near
#   zero or negative, which is common in distress. Total assets is
#   always positive (universe filter: assets >= €100k), so the ratio
#   is stable and comparable across company sizes.
#   Negative = equity eroding relative to the asset base,
#   a strong bankruptcy predictor.
#
# Both features are NaN in a company's first observed year and
# after a gap year — handled natively by XGBoost.

company_key = "company_id"   # unique per company (see step 1b)

# Sort so shift(1) always looks at the same company's previous year
df_features = df_features.sort_values([company_key, "Year"]).reset_index(drop=True)

grp = df_features.groupby(company_key)

# Prior-row values, kept only when that row is exactly Year − 1
prev_year      = grp["Year"].shift(1)
is_consecutive = (df_features["Year"] - prev_year) == 1

prior_assets = grp["Total assets"].shift(1).where(is_consecutive)
prior_equity = grp["Shareholders funds"].shift(1).where(is_consecutive)

df_features["FE_assets_yoy_growth"] = safe_divide(
    df_features["Total assets"] - prior_assets,
    prior_assets
)

df_features["FE_equity_yoy_change"] = safe_divide(
    df_features["Shareholders funds"] - prior_equity,
    prior_assets
)

# ── Check ─────────────────────────────────────────────────────
n_gaps = int((prev_year.notna() & ~is_consecutive).sum())
print(f"YoY features computed (grouped by {company_key})")
print(f"  Rows with a prior year:        {int(is_consecutive.sum()):,}")
print(f"  Rows after a gap (set to NaN): {n_gaps:,}")
print(f"  First observed year (NaN):     {int(prev_year.isna().sum()):,}")

# ── Validation ────────────────────────────────────────────────
fe_cols = [c for c in df_features.columns if c.startswith("FE_")]

print(f"Feature engineered columns added: {len(fe_cols)}")

print(f"\nNull counts per FE column:")
null_counts = df_features[fe_cols].isnull().sum()
for col, count in null_counts.items():
    pct = count / len(df_features) * 100
    print(f"  {col:<30} {count:>6,}  ({pct:.1f}%)")

print(f"\nDistress flag prevalence:")
print(f"  FE_negative_equity: {df_features['FE_negative_equity'].sum():,} "
      f"({df_features['FE_negative_equity'].mean():.1%})")
print(f"  FE_negative_pl:     {df_features['FE_negative_pl'].sum():,} "
      f"({df_features['FE_negative_pl'].mean():.1%})")

# Medians rather than means: financial figures are heavily skewed,
# so a few very large companies would dominate the means
print(f"\nBankrupt vs Active — median FE feature values:")
print(
    df_features.groupby("Bankrupt")[fe_cols]
    .median()
    .round(3)
    .T
    .rename(columns={0: "Active (median)", 1: "Bankrupt (median)"})
    .to_string()
)

print_df_structure("df_features", df_features)

# Model dataset: one row per company

The panel above has one row per company-year, so a company with three years of accounts appears three times. For modelling, each company is reduced to **one row: its latest year**, with its history carried in change features.

- **Each company counts once**, in training and in the metrics, regardless of how many years it reported.
- **The latest year is the most informative snapshot.** For a bankrupt company it is the last set of accounts before failure.
- **History is kept as features, not rows.** Year-over-year changes are computed on the full panel first, so the latest row still shows how the company was moving.
- **Calendar year is not a model feature.** Bankrupt companies stop filing, so their latest year is earlier than that of active companies. The year would reveal *when the records end*, not financial health.

In [ ]:
# ============================================================
# 9. MODEL DATASET — ONE ROW PER COMPANY (LATEST YEAR)
# ============================================================
# Step A adds two more change features on the full panel.
# Step B keeps each company's most recent year.
# Step C checks how much history is available per class.

# ── A: Change in key ratios over the latest year ─────────────
# Same rule as the YoY features in step 8F: compare only with
# the previous CALENDAR year of the same company; NaN after a gap
# or in a company's first observed year.
#
# FE_debt_ratio_change:
#   Debt ratio this year minus last year. Positive = leverage
#   rising, i.e. the balance sheet is weakening.
#
# FE_roa_change:
#   ROA this year minus last year. Negative = profitability
#   deteriorating relative to the asset base.
#
# Differences of ratios (not % changes): both ratios can be near
# zero or negative, where percentage changes break down.

df_features = df_features.sort_values(["company_id", "Year"]).reset_index(drop=True)
grp = df_features.groupby("company_id")

prev_year      = grp["Year"].shift(1)
is_consecutive = (df_features["Year"] - prev_year) == 1

df_features["FE_debt_ratio_change"] = (
    df_features["FE_debt_ratio"] - grp["FE_debt_ratio"].shift(1)
).where(is_consecutive)

df_features["FE_roa_change"] = (
    df_features["FE_roa"] - grp["FE_roa"].shift(1)
).where(is_consecutive)

# ── B: Keep each company's latest year ───────────────────────
# One row per company. The row keeps its Year for reference
# only — Year is not used as a model feature (see note above).
df_model = (
    df_features
    .sort_values(["company_id", "Year"])
    .groupby("company_id")
    .tail(1)
    .reset_index(drop=True)
)

# Years of data per company — reference only, not a feature
# (it would reveal how long a company kept filing)
years_per_company = df_features.groupby("company_id")["Year"].size()
df_model["n_years_reported"] = df_model["company_id"].map(years_per_company)

# Keep only what the model notebook needs, in a readable order:
# identifiers and label first, then reference columns, then the
# model inputs (raw figures, then engineered features).
# Dropped: Year of incorporation (replaced by Company Age) and the
# 4-digit NACE code (grouped into NACE Section).
model_columns = [
    # Identifiers and label
    "company_id",
    "Company name Latin alphabet",
    "Bankrupt",
    # Reference only — not model features
    "Year",
    "n_years_reported",
    # Company characteristics
    "NACE Section",
    "Company Age",
    # Reported figures (EUR)
    "Total assets",
    "Current assets",
    "Current liabilities",
    "Shareholders funds",
    "P/L before tax",
    # Engineered features
    "FE_total_liabilities",
    "FE_non_current_assets",
    "FE_debt_ratio",
    "FE_current_ratio",
    "FE_roa",
    "FE_negative_equity",
    "FE_negative_pl",
    "FE_assets_yoy_growth",
    "FE_equity_yoy_change",
    "FE_debt_ratio_change",
    "FE_roa_change",
]
df_model = df_model[model_columns]

assert df_model["company_id"].is_unique

# ── C: Summary and history check ─────────────────────────────
# Companies with only one year have no change features. If these
# are much more common among bankrupt companies, "missing history"
# itself could act as a risk signal — worth knowing before
# modelling and worth stating as a limitation.
n_act = int((df_model["Bankrupt"] == 0).sum())
n_bnk = int((df_model["Bankrupt"] == 1).sum())

print(f"Model dataset: {len(df_model):,} companies (one row each)")
print(f"  Active:   {n_act:,}")
print(f"  Bankrupt: {n_bnk:,}  ({n_bnk / len(df_model):.2%})")

print(f"\nLatest year of accounts, by class (% of companies):")
print(
    pd.crosstab(df_model["Bankrupt"], df_model["Year"], normalize="index")
    .mul(100).round(1)
    .rename(index={0: "Active", 1: "Bankrupt"})
    .to_string()
)

print(f"\nYears of data per company, by class (% of companies):")
print(
    pd.crosstab(df_model["Bankrupt"], df_model["n_years_reported"], normalize="index")
    .mul(100).round(1)
    .rename(index={0: "Active", 1: "Bankrupt"})
    .to_string()
)

change_cols = ["FE_assets_yoy_growth", "FE_equity_yoy_change",
               "FE_debt_ratio_change", "FE_roa_change"]
print(f"\nMissing change features, by class (% of companies):")
print(
    df_model.groupby("Bankrupt")[change_cols]
    .apply(lambda d: d.isna().mean() * 100)
    .round(1)
    .rename(index={0: "Active", 1: "Bankrupt"})
    .T
    .to_string()
)

# Inspect and save datasets

In [ ]:
# Display a sample of the engineered feature DataFrame to visually inspect the new features and their values across a subset of companies and years.

from IPython.display import HTML, display

sample = df_features.sample(10, random_state=1)
html = sample.to_html(index=False)

display(HTML(
    f'<div style="max-height:400px; overflow:auto; white-space:nowrap;">{html}</div>'
))

In [ ]:
# ============================================================
# 10. SAVE MODEL DATASET
# ============================================================
# Saves the modeling input — one row per company — to data/.
# The file is excluded from git: it contains company-level
# licensed data.
#
# Separating data preparation from modeling ensures:
#   - Notebook 2 can be re-run without reprocessing raw data
#   - The exact modeling input is versioned and reproducible
#   - Long Orbis Excel loading time is not repeated each run
#
# Float precision=6 retains sufficient decimal places for
# financial ratios without inflating file size.

output_path = os.path.join(data_dir, "model_data.csv")
df_model.to_csv(output_path, index=False, float_format="%.6f")

print(f"Saved: {output_path}")
print(f"  Rows:    {len(df_model):,} (one per company)")
print(f"  Columns: {len(df_model.columns)}")
print(f"  Size:    {os.path.getsize(output_path) / 1024**2:.2f} MB")


# ============================================================
# 11. EXPORT AGGREGATE DATA SUMMARIES FOR THE DASHBOARD
# ============================================================
# Writes column-level summaries (types, missing values, medians)
# to results/ for the Streamlit app. Only aggregates are saved —
# no company names or individual rows — so these files can be
# published while the Orbis data itself stays private.

results_dir = os.path.join(os.path.dirname(data_dir), "results")
os.makedirs(results_dir, exist_ok=True)

# ── A: Raw export — missing values by group ─────────────────
raw_summary = pd.DataFrame({
    "Column":               df_active_raw.columns,
    "Dtype":                df_active_raw.dtypes.astype(str).values,
    "Active non-null":      df_active_raw.notna().sum().values,
    "Active missing %":     (df_active_raw.isna().mean() * 100).round(1).values,
    "Bankrupt non-null":    df_bankrupt_raw.notna().sum().values,
    "Bankrupt missing %":   (df_bankrupt_raw.isna().mean() * 100).round(1).values,
})
raw_summary.to_csv(os.path.join(results_dir, "raw_columns_summary.csv"), index=False)

# ── B: Model dataset — missing values and group medians ────
num_cols_m = df_model.select_dtypes("number").columns
medians_m  = (
    df_model.groupby("Bankrupt")[num_cols_m]
    .median()
    .T
    .rename(columns={0: "Median active", 1: "Median bankrupt"})
)
model_summary = pd.DataFrame({
    "Column":    df_model.columns,
    "Dtype":     df_model.dtypes.astype(str).values,
    "Non-null":  df_model.notna().sum().values,
    "Missing %": (df_model.isna().mean() * 100).round(1).values,
}).merge(medians_m, left_on="Column", right_index=True, how="left")
# Identifier-like columns: medians are not meaningful
no_median = ["company_id", "Bankrupt", "Year", "n_years_reported"]
model_summary.loc[model_summary["Column"].isin(no_median),
                  ["Median active", "Median bankrupt"]] = np.nan
model_summary.to_csv(os.path.join(results_dir, "model_columns_summary.csv"), index=False)

# ── C: Headline counts ──────────────────────────────────────
company_labels_final = df_model.set_index("company_id")["Bankrupt"]
overview = pd.DataFrame({
    "Metric": ["Active companies (export)", "Bankrupt companies (export)",
               "Active companies (model data)", "Bankrupt companies (model data)",
               "Total companies (model data)",
               "Company-years of accounts", "First year", "Last year"],
    "Value":  [len(df_active_raw), len(df_bankrupt_raw),
               int((company_labels_final == 0).sum()),
               int((company_labels_final == 1).sum()),
               len(company_labels_final),
               len(df_features), int(df_features["Year"].min()),
               int(df_features["Year"].max())],
})
overview.to_csv(os.path.join(results_dir, "dataset_overview.csv"), index=False)

print("Saved to results/: raw_columns_summary.csv, model_columns_summary.csv, "
      "dataset_overview.csv")